## WARNING: NOTEBOOK SYNC RULE
After modifying any src/ file, update this notebook to reflect those changes.
Keep `redqueen.ipynb` (local dev) in sync with `notebooks/train_on_kaggle.ipynb` (Kaggle).

**Last synced: 2026-06-10**

**Changes on 2026-06-10:**
- Added Environment Validation cell (between Cell 2 and Phase 1 header)
- reward.py updated to v5: smooth late-game ramp, kill-assist, zone control, wasted-bomb penalty
- ppo_trainer.py: VecNormalize added for reward normalisation (stats saved per stage as `vecnormalize_{stage}.pkl`)
- Cell 6 now explicitly uses `--init-from-tactical` (not `--init-from`)
- Cell 1: added Kaggle pip-install reminder comment

**Pipeline (11 cells):**
1. Cell 1 — Local environment setup + repo path injection
2. Cell 2 — Configure output directories
3. Cell ENV — Environment validation (import sanity checks)
4. Cell 3 (markdown) — Phase 1: TacticalAgent Behavioral Cloning header
5. TacticalBC Cell A — Generate BC data via TacticalAgent self-rollout
6. TacticalBC Cell B — Train BC on TacticalAgent data
7. Cell 5 — Resolve TacticalBC best checkpoint path
8. Cell 6 — Phase 3: PPO Curriculum (7 stages)
9. Cell 7 — Phase 4: Self-play (optional)
10. Cell 8 — Export: ONNX (primary) + TorchScript (fallback)
11. Cell 9 — Prepare submission folder (3 files: `agent.py`, `model.onnx`, `model.pt`)

**Key invariants:**
- `agent.py` must be at ZIP ROOT — never inside a subfolder
- Never import PyTorch inside `agent.py` — ONNX Runtime only
- Bomb pad to MAX_BOMBS=16 before ONNX export
- Do NOT include `requirements.txt` in submission.zip — evaluator rejects it
- VecNormalize stats saved to `vecnormalize_{stage}.pkl` alongside checkpoints

In [ ]:
# ── Cell 1: Local environment setup ───────────────────────────────────────────
# Kaggle: add pip install stable-baselines3 sb3-contrib here

import os
import sys
from pathlib import Path

# Suppress TF/XLA/gRPC C++ noise
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GLOG_minloglevel"]      = "3"
os.environ["GRPC_VERBOSITY"]        = "ERROR"

import warnings
warnings.filterwarnings("ignore", message=".*shared layers.*", category=UserWarning)

# Repo root — adjust if running from a different location
REPO_DIR = Path("__file__").resolve().parent
if not (REPO_DIR / "agent").exists():
    # Fallback: assume CWD is the repo root
    REPO_DIR = Path.cwd()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repo root:", REPO_DIR)
print("Python:", sys.version)

In [ ]:
# ── Cell 2: Configure output directories ──────────────────────────────────────
from pathlib import Path

REPO_DIR        = Path("__file__").resolve().parent
if not (REPO_DIR / "agent").exists():
    REPO_DIR = Path.cwd()

ARTIFACTS_DIR   = REPO_DIR / "training_artifacts"
CKPT_DIR        = ARTIFACTS_DIR / "checkpoints"
DATA_DIR        = ARTIFACTS_DIR / "data"
LOGS_DIR        = ARTIFACTS_DIR / "logs"
PAST_AGENTS_DIR = CKPT_DIR / "past_agents"
SUBMISSION_DIR  = REPO_DIR / "submissions" / "latest"

for d in [CKPT_DIR, DATA_DIR, LOGS_DIR, PAST_AGENTS_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Artifacts dir:", ARTIFACTS_DIR)
print("Submission dir:", SUBMISSION_DIR)

In [ ]:
# ── Environment Validation: sanity-check src/ imports ─────────────────────────
## Environment Validation

import subprocess, sys

checks = [
    ("reward",   "from src.training.reward import compute_reward; print('reward OK')"),
    ("features", "from src.utils.feature_extractor import extract_features; print('features OK')"),
    ("network",  "from src.models.policy_network import BomberPolicyNet; print('network OK')"),
]

all_ok = True
for name, stmt in checks:
    result = subprocess.run(
        [sys.executable, "-c", stmt],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(result.stdout.strip())
    else:
        print(f"FAIL ({name}):", result.stderr.strip())
        all_ok = False

assert all_ok, "One or more src/ imports failed — fix before running training."

## Phase 1: TacticalAgent Behavioral Cloning

In [ ]:
# ── TacticalBC Cell A: Generate BC data via TacticalAgent self-rollout ─────────
import os
import sys
from pathlib import Path

REPO_DIR  = Path("__file__").resolve().parent
if not (REPO_DIR / "agent").exists():
    REPO_DIR = Path.cwd()
ARTIFACTS_DIR = REPO_DIR / "training_artifacts"
DATA_DIR  = ARTIFACTS_DIR / "data"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ── Configurable variables ─────────────────────────────────────────────────────
N_GAMES      = 200       # number of self-rollout games to generate
DATA_DIR_STR = str(DATA_DIR)

%cd {REPO_DIR}
!python -m src.training.tactical_bc \
    --generate \
    --n-games {N_GAMES} \
    --data-dir {DATA_DIR_STR}

print(f"TacticalBC data generation complete: {DATA_DIR_STR}")

In [ ]:
# ── TacticalBC Cell B: Train Behavioral Cloning on TacticalAgent data ──────────
import os
import sys
from pathlib import Path

REPO_DIR  = Path("__file__").resolve().parent
if not (REPO_DIR / "agent").exists():
    REPO_DIR = Path.cwd()
ARTIFACTS_DIR = REPO_DIR / "training_artifacts"
DATA_DIR  = ARTIFACTS_DIR / "data"
CKPT_DIR  = ARTIFACTS_DIR / "checkpoints"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ── Configurable variables ─────────────────────────────────────────────────────
BC_EPOCHS    = 20
DATA_DIR_STR = str(DATA_DIR)
CKPT_DIR_STR = str(CKPT_DIR)

%cd {REPO_DIR}
!python -m src.training.tactical_bc \
    --train \
    --epochs {BC_EPOCHS} \
    --data-dir {DATA_DIR_STR} \
    --output-dir {CKPT_DIR_STR}

print(f"TacticalBC training complete. Checkpoints in: {CKPT_DIR_STR}")

In [ ]:
# ── Cell 5: Resolve TacticalBC best checkpoint ─────────────────────────────────
# Finds the most-recently modified tactical_bc checkpoint in CKPT_DIR.
# Set TACTICAL_BC_CKPT manually below if you want a specific checkpoint.

import os
import sys
from pathlib import Path

REPO_DIR      = Path("__file__").resolve().parent
if not (REPO_DIR / "agent").exists():
    REPO_DIR = Path.cwd()
ARTIFACTS_DIR = REPO_DIR / "training_artifacts"
CKPT_DIR      = ARTIFACTS_DIR / "checkpoints"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ── Configurable variables ─────────────────────────────────────────────────────
# Set this to a specific checkpoint path, or leave empty for auto-resolve.
CHECKPOINT = ""   # e.g. str(CKPT_DIR / "tactical_bc_20ep_20260529_120000.pt")

if not CHECKPOINT:
    candidates = sorted(
        CKPT_DIR.glob("tactical_bc_*.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    assert candidates, (
        f"No tactical_bc_*.pt found in {CKPT_DIR}. "
        "Run TacticalBC Cell B first, or set CHECKPOINT manually."
    )
    CHECKPOINT = str(candidates[0])

TACTICAL_BC_CKPT = CHECKPOINT
print(f"Using TacticalBC checkpoint: {TACTICAL_BC_CKPT}")

In [ ]:
# ── Cell 6: Phase 3 — PPO curriculum training ──────────────────────────────────
#
# Curriculum stages (7 stages, each requires min 100k steps before advancing):
#   Stage 0: random          avg_rank <= 0.8
#   Stage 1: simple          avg_rank <= 1.2
#   Stage 2: simple_smarter1 avg_rank <= 1.5  (BRIDGE: 1 Smarter + 2 Simple)
#   Stage 3: smarter         avg_rank <= 1.8
#   Stage 4: tactical        avg_rank <= 2.0
#   Stage 5: trapper         avg_rank <= 2.0
#   Stage 6: genius          avg_rank <= 2.0
#
# ent_coef schedule: 0.08 -> 0.10 -> 0.08 -> 0.07 -> 0.06 -> 0.05 -> 0.04
# LR schedule:       3e-4 -> 1.5e-4 -> 1.2e-4 -> 1e-4 -> 8e-5 -> 8e-5 -> 5e-5
#   (optimizer state cleared at each stage transition)
#
# Stage 0->1 special: reload BC (not Stage 0 best) + clear optimizer state.
# --init-from-tactical passes the tactical BC checkpoint for Stage 0->1 re-anchor.
# Rolling mean advancement: last 3 eval windows <= threshold AND min 100k steps.
# Evaluation: every EVAL_FREQ steps over EVAL_EPISODES episodes.
# 20% random opponent mixing per training env (prevents co-adaptation).
#
# VecNormalize (reward normalisation only, norm_obs=False):
#   - Running mean/var of rewards saved to {CKPT_DIR}/vecnormalize_{stage}.pkl
#     at every stage transition so the normaliser is not reset mid-curriculum.
#   - On resume/restart, the matching vecnormalize_{stage}.pkl is auto-loaded.

import os
import sys
from pathlib import Path

REPO_DIR       = Path("__file__").resolve().parent
if not (REPO_DIR / "agent").exists():
    REPO_DIR = Path.cwd()
ARTIFACTS_DIR  = REPO_DIR / "training_artifacts"
CKPT_DIR       = ARTIFACTS_DIR / "checkpoints"
LOGS_DIR       = ARTIFACTS_DIR / "logs"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ── Configurable variables — set these before running ─────────────────────────
# Set CHECKPOINT to the tactical BC checkpoint from Cell 5, or leave empty for auto-resolve.
CHECKPOINT           = ""         # e.g. str(CKPT_DIR / "tactical_bc_20ep_20260529.pt")
PPO_STEPS_PER_STAGE  = 750_000    # max steps per curriculum stage
N_ENVS               = 4          # parallel envs
EVAL_FREQ            = 50_000     # evaluate every N steps
EVAL_EPISODES        = 200        # episodes per evaluation window
MIN_STEPS_PER_STAGE  = 100_000    # minimum steps before stage can advance
DEVICE               = "auto"     # "cuda", "cpu", or "auto"

CKPT_DIR_STR = str(CKPT_DIR)
LOGS_DIR_STR = str(LOGS_DIR)

# Auto-resolve tactical BC checkpoint if CHECKPOINT not set manually
if not CHECKPOINT:
    candidates = sorted(
        CKPT_DIR.glob("tactical_bc_*.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    assert candidates, (
        f"No tactical_bc_*.pt found in {CKPT_DIR}. "
        "Run TacticalBC Cell B and Cell 5 first, or set CHECKPOINT manually."
    )
    CHECKPOINT = str(candidates[0])

TACTICAL_BC_CKPT = CHECKPOINT
print(f"PPO init checkpoint:  {TACTICAL_BC_CKPT}")
print(f"Steps per stage:      {PPO_STEPS_PER_STAGE:,}")
print(f"Parallel envs:        {N_ENVS}")
print(f"VecNormalize stats:   {CKPT_DIR_STR}/vecnormalize_{{stage}}.pkl")

%cd {REPO_DIR}
!python -m src.training.ppo_trainer \
    --curriculum \
    --init-from-tactical {TACTICAL_BC_CKPT} \
    --output-dir {CKPT_DIR_STR} \
    --log-dir {LOGS_DIR_STR} \
    --total-steps-per-stage {PPO_STEPS_PER_STAGE} \
    --n-envs {N_ENVS} \
    --eval-freq {EVAL_FREQ} \
    --eval-episodes {EVAL_EPISODES} \
    --min-steps-per-stage {MIN_STEPS_PER_STAGE} \
    --device {DEVICE}

print("PPO curriculum complete.")

In [ ]:
# ── Cell 7 (optional): Phase 4 — Self-play ─────────────────────────────────────
# Runs only if curriculum (Cell 6) passed ALL 7 stages.
#
# Trains against a rolling pool of the last 20 past snapshots,
# sampling recent ones more often (exponential-decay weights, alpha=0.9^age).

import os
import sys
from pathlib import Path

REPO_DIR        = Path("__file__").resolve().parent
if not (REPO_DIR / "agent").exists():
    REPO_DIR = Path.cwd()
ARTIFACTS_DIR   = REPO_DIR / "training_artifacts"
CKPT_DIR        = ARTIFACTS_DIR / "checkpoints"
PAST_AGENTS_DIR = CKPT_DIR / "past_agents"
LOGS_DIR        = ARTIFACTS_DIR / "logs"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ── Configurable variables — set these before running ─────────────────────────
# Set CHECKPOINT to the best PPO curriculum checkpoint, or leave empty for auto-resolve.
CHECKPOINT          = ""        # e.g. str(CKPT_DIR / "ppo_curriculum_best.pt")
SELFPLAY_STEPS      = 500_000   # total self-play steps
N_ENVS              = 4         # parallel envs
SNAPSHOT_EVERY      = 50_000    # snapshot interval (steps)
POOL_SIZE           = 20        # max past-agent pool size
DEVICE              = "auto"    # "cuda", "cpu", or "auto"

CKPT_DIR_STR        = str(CKPT_DIR)
PAST_AGENTS_DIR_STR = str(PAST_AGENTS_DIR)
LOGS_DIR_STR        = str(LOGS_DIR)

PAST_AGENTS_DIR.mkdir(parents=True, exist_ok=True)

# Auto-resolve PPO best checkpoint if CHECKPOINT not set manually
if not CHECKPOINT:
    candidates = sorted(
        list(CKPT_DIR.glob("ppo_*.pt")) + list(CKPT_DIR.glob("selfplay_*.pt")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        CHECKPOINT = str(candidates[0])
    else:
        print("WARNING: No PPO checkpoint found — skipping self-play.")
        CHECKPOINT = None

PPO_BEST_CKPT = CHECKPOINT

if PPO_BEST_CKPT:
    print(f"Self-play init checkpoint: {PPO_BEST_CKPT}")
    print(f"Total self-play steps:     {SELFPLAY_STEPS:,}")

    %cd {REPO_DIR}
    !python -m src.training.ppo_trainer \
        --self-play \
        --init-from {PPO_BEST_CKPT} \
        --output-dir {CKPT_DIR_STR} \
        --snapshot-dir {PAST_AGENTS_DIR_STR} \
        --log-dir {LOGS_DIR_STR} \
        --total-steps {SELFPLAY_STEPS} \
        --n-envs {N_ENVS} \
        --snapshot-every {SNAPSHOT_EVERY} \
        --pool-size {POOL_SIZE} \
        --device {DEVICE}

    print("Self-play complete.")
else:
    print("Self-play skipped — no PPO checkpoint available.")

In [ ]:
# ── Cell 8: Export ONNX + TorchScript ─────────────────────────────────────────
# Produces two model files:
#   model.onnx — primary inference via onnxruntime (fastest, required)
#   model.pt   — TorchScript fallback (torch.jit.load, if onnxruntime unavailable)
#
# Bomb array is padded to MAX_BOMBS=16 for fixed ONNX input shapes.

import os
import sys
from pathlib import Path

REPO_DIR      = Path("__file__").resolve().parent
if not (REPO_DIR / "agent").exists():
    REPO_DIR = Path.cwd()
ARTIFACTS_DIR = REPO_DIR / "training_artifacts"
CKPT_DIR      = ARTIFACTS_DIR / "checkpoints"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ── Configurable variables — set these before running ─────────────────────────
# Set CHECKPOINT to the checkpoint to export, or leave empty for auto-resolve.
CHECKPOINT  = ""     # e.g. str(CKPT_DIR / "selfplay_best.pt")
ONNX_OPSET  = 17     # ONNX opset version
VERIFY_ONNX = True   # run a forward-pass sanity check after export

ONNX_OUTPUT = str(CKPT_DIR / "model.onnx")
PT_OUTPUT   = str(CKPT_DIR / "model.pt")

# Auto-resolve best checkpoint (selfplay > ppo > tactical_bc, most recent wins)
if not CHECKPOINT:
    for pattern in ("selfplay_*.pt", "ppo_*.pt", "tactical_bc_*.pt"):
        candidates = sorted(
            CKPT_DIR.glob(pattern),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if candidates:
            CHECKPOINT = str(candidates[0])
            break

assert CHECKPOINT, f"No checkpoint found in {CKPT_DIR}. Set CHECKPOINT manually."
BEST_CKPT = CHECKPOINT
print(f"Exporting checkpoint:  {BEST_CKPT}")
print(f"ONNX output:           {ONNX_OUTPUT}")
print(f"TorchScript output:    {PT_OUTPUT}")

%cd {REPO_DIR}
!python -m src.utils.export_onnx \
    --checkpoint {BEST_CKPT} \
    --output {ONNX_OUTPUT} \
    --opset {ONNX_OPSET} \
    {"--verify" if VERIFY_ONNX else ""}

!python -m src.utils.export_onnx \
    --checkpoint {BEST_CKPT} \
    --output {PT_OUTPUT} \
    --torchscript

print("Export complete.")

In [ ]:
# ── Cell 9: Prepare submission folder ──────────────────────────────────────────
#
# Competition format — submission.zip must contain exactly these 3 files at root:
#   agent.py    <- MANDATORY entry point, must be at root (NOT inside a subfolder)
#   model.onnx  <- primary ONNX inference
#   model.pt    <- TorchScript fallback
#
# DO NOT include requirements.txt — the evaluator rejects it (requirements_txt_forbidden).

import shutil
import zipfile
import time
from pathlib import Path

REPO_DIR        = Path("__file__").resolve().parent
if not (REPO_DIR / "agent").exists():
    REPO_DIR = Path.cwd()
ARTIFACTS_DIR   = REPO_DIR / "training_artifacts"
CKPT_DIR        = ARTIFACTS_DIR / "checkpoints"
SUBMISSION_DIR  = REPO_DIR / "submissions" / "latest"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# ── Configurable variables ─────────────────────────────────────────────────────
AGENT_SRC  = REPO_DIR / "agent" / "agent.py"
ONNX_SRC   = CKPT_DIR / "model.onnx"
PT_SRC     = CKPT_DIR / "model.pt"

AGENT_DST  = SUBMISSION_DIR / "agent.py"
ONNX_DST   = SUBMISSION_DIR / "model.onnx"
PT_DST     = SUBMISSION_DIR / "model.pt"

for src, dst in [(AGENT_SRC, AGENT_DST), (ONNX_SRC, ONNX_DST), (PT_SRC, PT_DST)]:
    assert src.exists(), f"FATAL: source file missing: {src}"
    shutil.copy2(src, dst)
    print(f"Copied: {src.name} -> {dst}")

# Validate no requirements.txt in submission
assert not (SUBMISSION_DIR / "requirements.txt").exists(), \
    "FATAL: requirements.txt is FORBIDDEN — remove it!"

# Create timestamped zip
ts = time.strftime("%Y%m%d_%H%M")
zip_path = REPO_DIR / "submissions" / f"submission_{ts}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in [AGENT_DST, ONNX_DST, PT_DST]:
        zf.write(f, f.name)   # store at root, not inside a subfolder

# Verify agent.py is at root
with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()

print("\nFiles in submission zip:")
for n in sorted(names):
    print(f"  {n}")

assert "agent.py" in names, "FATAL: agent.py not at root of submission.zip!"
print("\nagent.py at zip root:     OK")
print("model.onnx present:       OK")
print("model.pt present:         OK")
print("requirements.txt absent:  OK")
print(f"\nSubmission zip: {zip_path}")